# Alexandria Retrieve Smoke Notebook

This notebook is a manual smoke check for the real retrieval path. It uses the configured database and configured provider adapters through the CLI and `App.retrieve(...)`, then inspects persisted state with SQLAlchemy.

It is not the only validation for retrieval behavior; the matching pytest command is at the end.


## Prerequisites

Expected setup:
- Docker and Task are available locally.
- Provider credentials are present in `.env` or the environment, for example `ALEXANDRIA_EMBEDDING__API_KEY` and `ALEXANDRIA_SUMMARIZER__API_KEY` for OpenAI-backed adapters.
- The notebook is run from the repository root or from the `sandbox/` directory.

Expected output note: the setup cell starts Postgres/pgvector through `task deploy`. The ingest cell should print document ids and source keys. If infrastructure or credentials are missing, the smoke path should fail before retrieval inspection.


## Start Infrastructure

This cell starts the local infrastructure profile through the project Taskfile and waits for Docker Compose health checks before the smoke path runs.


In [ ]:
from __future__ import annotations

from pathlib import Path
import subprocess


repo = Path.cwd()
if not (repo / "pyproject.toml").exists():
    repo = repo.parent

deploy = subprocess.run(
    ["task", "deploy", "--", "--wait"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

if deploy.stdout:
    print(deploy.stdout)
if deploy.stderr:
    print(deploy.stderr)


## Ingest Retrieval Fixtures Through The CLI

This cell creates two real documents through the public CLI entrypoint. The source keys are timestamped so the persisted rows can be found again without relying on a clean database.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import subprocess


repo = Path.cwd()
if not (repo / "pyproject.toml").exists():
    repo = repo.parent

stamp = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
docs = [
    {
        "name": "Notebook Retrieve Alpha",
        "body": "Alpha retrieval smoke document about semantic search and vector routing.",
        "source_key": f"notebook:retrieve:alpha:{stamp}",
    },
    {
        "name": "Notebook Retrieve Beta",
        "body": "Beta retrieval smoke document about background jobs and split checks.",
        "source_key": f"notebook:retrieve:beta:{stamp}",
    },
]

ingested: dict[str, str] = {}
for doc in docs:
    result = subprocess.run(
        [
            "uv",
            "run",
            "cli",
            "ingest",
            "--name",
            doc["name"],
            "--body",
            doc["body"],
            "--source-key",
            doc["source_key"],
        ],
        cwd=repo,
        text=True,
        capture_output=True,
        check=True,
    )
    ingested[doc["source_key"]] = result.stdout.strip()

print("ingested")
for source_key, document_id in ingested.items():
    print(" ", source_key, document_id)


## Retrieve Through The Real App Boundary

This cell uses `get_app()` and the configured infrastructure dependencies. It does not patch app factories or construct fake ports.


In [ ]:
from application.app import get_app
from infrastructure.config import get_settings


get_settings.cache_clear()
get_app.cache_clear()
app = get_app()

query = "alpha semantic retrieval smoke query"
hits = await app.retrieve(query, limit=5)

print("query:", query)
print("hit_count:", len(hits))
for hit in hits:
    print(
        "hit",
        {
            "doc_id": str(hit.doc.id),
            "source_key": hit.doc.source_key,
            "name": hit.doc.name,
            "score": round(hit.score, 6),
            "distance": round(hit.distance or 0.0, 6),
            "bm25": hit.bm25,
        },
    )

assert hits, "retrieve returned no hits"
assert any(hit.doc.source_key in ingested for hit in hits), (
    "retrieve did not return any smoke document"
)


## Inspect Persisted State

Expected notes:
- Both document rows should exist with the CLI source keys, generated summaries, and provider embeddings.
- Embedding previews should show bounded numeric vectors without dumping the full embeddings.
- Retrieved hits should point at persisted document rows and active leaves.


In [ ]:
from sqlalchemy import select

from domain.entity import Document, Node
from infrastructure.config import get_settings
from infrastructure.persistence.db import Db


get_settings.cache_clear()
settings = get_settings()
db = Db(settings.database)
source_keys = set(ingested)

with db.session() as session:
    documents = session.scalars(
        select(Document)
        .where(Document.source_key.in_(source_keys))
        .order_by(Document.source_key.asc())
    ).all()
    assert len(documents) == len(source_keys), "not all retrieval smoke documents were persisted"

    for document in documents:
        leaf = session.get(Node, document.leaf_id)
        embedding = list(document.embedding)
        preview = [round(value, 6) for value in embedding[:20]]
        print("document")
        print("  id:", document.id)
        print("  source_key:", document.source_key)
        print("  name:", document.name)
        print("  summary:", document.summary)
        print("  leaf_id:", document.leaf_id)
        print("  leaf_status:", leaf.status if leaf else None)
        print("  embedding dimensions:", len(embedding))
        print("  embedding first 20:", preview)


### Expected Smoke Output

- `hit_count` should be greater than zero.
- At least one hit should have one of the timestamped notebook source keys.
- Embedding dimensions should match the configured embedding adapter output dimensions.
- Document rows should be attached to active leaves unless another worker changed the tree while the notebook ran.


### Matching Pytest Smoke Command

Run this from the repository root:

```bash
uv run pytest tests/integration/test_retrieve_flow.py -q
```


## Tear Down Infrastructure

Run this final cell when the smoke check is complete. It stops the local infrastructure profile through the project Taskfile without deleting Docker volumes.


In [ ]:
from pathlib import Path
import subprocess


repo = Path.cwd()
if not (repo / "pyproject.toml").exists():
    repo = repo.parent

teardown = subprocess.run(
    ["task", "teardown"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

if teardown.stdout:
    print(teardown.stdout)
if teardown.stderr:
    print(teardown.stderr)
